# A trait-derived single-tile ecosystem

This notebook simulates isolated food webs whose species are authored entirely as sets of traits. A deterministic compiler turns those traits into numerical population parameters and diet relationships before the fast numerical solver runs.

Populations are total biomass within one land area, not individual counts. Size-class traits provide individual biomass, making approximate individual counts available when needed. There is no migration or ecosystem-specific calibration. Each tile receives the twelve monthly temperature, insolation, and precipitation samples produced by the Planet climate simulator.

In [ ]:
%use kandy

In [ ]:
import kotlin.math.PI
import kotlin.math.abs
import kotlin.math.log10
import kotlin.math.pow
import kotlin.math.roundToInt
import kotlin.math.sin
import kotlin.math.sqrt
import kotlin.random.Random
import dev.biserman.planet.planet.climate.ClimateDatum
import dev.biserman.planet.planet.climate.ClimateDatumSample
import dev.biserman.planet.planet.climate.ClimateClassification
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.line
import dev.biserman.planet.planet.ecology.*


## Shared ecology model

Traits, compiled species, taxonomy, feeding dynamics, suitability, and the numerical simulator now live in dev.biserman.planet.planet.ecology. This notebook keeps climate fixtures, validation probes, and exploratory simulation runs.


In [ ]:
/** Builds an illustrative January-to-December climate in native ecology units. */
fun sampleClimate(
    tileId: Int,
    temperaturesC: List<Double>,
    insolationsWm2: List<Double>,
    precipitationMm: List<Double>,
): ClimateDatum {
    require(temperaturesC.size == 12)
    require(insolationsWm2.size == 12)
    require(precipitationMm.size == 12)
    return ClimateDatum(
        tileId = tileId,
        months = temperaturesC.indices.map { month ->
            ClimateDatumSample(
                temperaturesC[month],
                insolationsWm2[month],
                precipitationMm[month],
            )
        },
    )
}

/** Hot subtropical desert with intense light and very sparse rainfall. */
val desertClimate = sampleClimate(
    tileId = -2,
    temperaturesC = listOf(18.0, 20.0, 24.0, 29.0, 33.0, 36.0, 38.0, 37.0, 34.0, 29.0, 23.0, 19.0),
    insolationsWm2 = listOf(220.0, 245.0, 280.0, 310.0, 330.0, 345.0, 340.0, 325.0, 300.0, 270.0, 235.0, 215.0),
    precipitationMm = listOf(5.0, 4.0, 4.0, 2.0, 1.0, 0.5, 1.0, 2.0, 3.0, 5.0, 6.0, 6.0),
)

/** Tropical savanna with a pronounced summer wet season and winter drought. */
val savannaClimate = sampleClimate(
    tileId = -3,
    temperaturesC = listOf(24.0, 25.0, 27.0, 29.0, 29.0, 28.0, 27.0, 27.0, 28.0, 28.0, 26.0, 24.0),
    insolationsWm2 = listOf(235.0, 250.0, 270.0, 280.0, 265.0, 235.0, 220.0, 225.0, 245.0, 265.0, 255.0, 235.0),
    precipitationMm = listOf(8.0, 12.0, 25.0, 65.0, 130.0, 190.0, 220.0, 185.0, 110.0, 45.0, 15.0, 8.0),
)

/** Equatorial rainforest with warm temperatures and abundant rain year-round. */
val jungleClimate = sampleClimate(
    tileId = -4,
    temperaturesC = listOf(26.0, 26.0, 26.5, 26.5, 26.5, 26.0, 25.5, 26.0, 26.5, 26.5, 26.0, 26.0),
    insolationsWm2 = listOf(205.0, 215.0, 220.0, 215.0, 200.0, 185.0, 190.0, 205.0, 220.0, 225.0, 215.0, 205.0),
    precipitationMm = listOf(240.0, 220.0, 260.0, 285.0, 300.0, 245.0, 210.0, 205.0, 225.0, 275.0, 290.0, 260.0),
)

/** Continental boreal forest with a long cold winter and short mild summer. */
val borealClimate = sampleClimate(
    tileId = -5,
    temperaturesC = listOf(-18.0, -15.0, -8.0, 1.0, 9.0, 15.0, 18.0, 15.0, 8.0, 0.0, -9.0, -16.0),
    insolationsWm2 = listOf(25.0, 55.0, 110.0, 175.0, 225.0, 255.0, 245.0, 190.0, 125.0, 65.0, 30.0, 15.0),
    precipitationMm = listOf(25.0, 20.0, 22.0, 28.0, 40.0, 60.0, 75.0, 65.0, 50.0, 38.0, 30.0, 25.0),
)

/** Arctic tundra with a brief cool growing season and low precipitation. */
val tundraClimate = sampleClimate(
    tileId = -6,
    temperaturesC = listOf(-26.0, -25.0, -20.0, -11.0, -3.0, 4.0, 8.0, 6.0, 1.0, -8.0, -18.0, -24.0),
    insolationsWm2 = listOf(2.0, 18.0, 75.0, 145.0, 205.0, 235.0, 215.0, 160.0, 90.0, 32.0, 5.0, 0.5),
    precipitationMm = listOf(10.0, 8.0, 8.0, 9.0, 12.0, 18.0, 25.0, 24.0, 18.0, 14.0, 11.0, 10.0),
)

/** Permanent ice cap with no month above freezing and polar-light seasonality. */
val iceCapClimate = sampleClimate(
    tileId = -7,
    temperaturesC = listOf(-38.0, -40.0, -38.0, -31.0, -23.0, -17.0, -14.0, -17.0, -24.0, -31.0, -35.0, -37.0),
    insolationsWm2 = listOf(0.0, 2.0, 35.0, 100.0, 165.0, 205.0, 190.0, 125.0, 55.0, 10.0, 0.0, 0.0),
    precipitationMm = listOf(5.0, 4.0, 4.0, 4.0, 5.0, 7.0, 9.0, 8.0, 7.0, 6.0, 5.0, 5.0),
)

/** Drop-in named climate presets for notebook ecosystem experiments. */
val sampleClimates: Map<String, ClimateDatum> = mapOf(
    "desert" to desertClimate,
    "savanna" to savannaClimate,
    "jungle" to jungleClimate,
    "boreal" to borealClimate,
    "tundra" to tundraClimate,
    "ice_cap" to iceCapClimate,
)


## Feeding and population change

Feeding fluxes and population derivatives are imported from the shared ecology package.


In [ ]:
// Feeding and population dynamics are imported from dev.biserman.planet.planet.ecology.


## Trait compilation and species catalog

Trait compilation, the production species catalog, taxonomic orders, and Hersfeldt mappings are imported from the shared ecology package. The following cell retains notebook-specific regression probes and the sample ecosystem selection.


In [ ]:
// Trait compilation and catalog definitions are imported from dev.biserman.planet.planet.ecology.


In [ ]:
val mesquiteClimateNiche = speciesCatalog.first { it.id == "mesquite" }.climateNiche
val cactusClimateNiche = speciesCatalog.first { it.id == "cactus" }.climateNiche
check(mesquiteClimateNiche.maxOptimalMoistureMm == 140.0) {
    "Drought-deciduous mesquite should be penalized by rainforest rainfall"
}
check(cactusClimateNiche.maxOptimalMoistureMm == 80.0) {
    "CAM cactus should be strongly waterlogging-sensitive"
}
check(Trait.LARGE_CANOPY in speciesCatalog.first { it.id == "kapok_tree" }.traits)
check(speciesCatalog.first { it.id == "frog" }.let { frog ->
    Trait.WETLAND_BREEDING in frog.traits && Trait.AQUATIC_LARVAE in frog.traits
}) { "Frogs should derive both wetland dependence and wetland refuge from traits" }

val habitatProbeSpecies = speciesCatalog.filter {
    it.id in setOf("grass", "kapok_tree", "frog", "zebra")
}
val habitatProbeModel = EcosystemModel(
    species = habitatProbeSpecies,
    seasonsEnabled = false,
    landArea = 100.0,
    climate = jungleClimate,
)
val habitatProbeState = habitatProbeSpecies.associate { definition ->
    definition.id to if (definition.id == "kapok_tree") {
        habitatProbeModel.baseSessileCarryingCapacityDensity(SizeClass.GIANT) *
            habitatProbeModel.landArea
    } else 10.0 * definition.individualBiomass
}
val habitatProbeValues = habitatAxisValues(
    0.0,
    habitatProbeState,
    habitatProbeModel,
)
check(availableLightFraction(
    speciesCatalog.first { it.id == "grass" },
    0.0,
    habitatProbeState,
    habitatProbeModel,
) <= 0.051) { "A full kapok canopy should leave about five percent ground light" }
check(habitatStressMortalityRate(
    speciesCatalog.first { it.id == "zebra" },
    habitatProbeValues,
) > 0.9) { "Closed canopy should be strongly unsuitable for open-plains locomotion" }
check(habitatRefugePreferenceMultiplier(
    speciesCatalog.first { it.id == "frog" },
    habitatProbeValues,
) <= 0.31) { "Wetland habitat should provide frogs a strong predation refuge" }

/** Representative woody producers supplied for each terrestrial sample biome. */
val representativeWoodyPlantsByBiome = mapOf(
    "desert" to setOf("mesquite"),
    "savanna" to setOf("acacia"),
    "jungle" to setOf("kapok_tree"),
    "oceanic_temperate" to setOf("oak"),
    "boreal" to setOf("spruce"),
    "tundra" to setOf("dwarf_willow"),
)
check(representativeWoodyPlantsByBiome.values.flatten().all { plantId ->
    speciesCatalog.any { it.id == plantId && Trait.WOODY in it.traits }
}) { "Every terrestrial biome example should have a woody producer" }

val frogPreyIds = speciesCatalog.first { it.id == "frog" }.diet.keys
    .filter { it.resource == FoodResource.MOTILE_PREY }
    .map { it.preyId }
    .toSet()
check(frogPreyIds.containsAll(setOf("grasshopper", "caterpillar", "cricket", "beetle", "fruit_fly"))) {
    "Frogs should have several catalog insect prey options"
}
val bambooDefinition = speciesCatalog.first { it.id == "bamboo" }
check(bambooDefinition.blueprint.hasTag(EffectTag.WINTER_SURVIVAL)) {
    "Bamboo rhizomes should protect it through temperate winters"
}
check(bambooDefinition.climateNiche.minOptimalMoistureMm >= 70.0) {
    "Bamboo should require a high-precipitation environment"
}
check(bambooDefinition.climateNiche.maxOptimalTemperatureC -
    bambooDefinition.climateNiche.minOptimalTemperatureC >= 35.0
) { "Bamboo should tolerate a broad temperature range" }
val giantPandaDiet = speciesCatalog.first { it.id == "giant_panda" }.diet.keys
check(giantPandaDiet == setOf(FoodSource("bamboo", FoodResource.BAMBOO))) {
    "Giant pandas should derive a near-exclusive bamboo diet"
}
val toucanFruitSources = speciesCatalog.first { it.id == "toucan" }.diet.keys
check(toucanFruitSources.all { it.resource == FoodResource.FRUIT } &&
    toucanFruitSources.map { it.preyId }.containsAll(
        setOf("apple_tree", "mango_tree", "fig_tree", "date_palm"),
    )
) { "Toucans should derive fruit pathways from every catalog fruit tree" }

val overbuiltChimeraBlueprint = speciesBlueprint(
    "overbuilt_chimera_test",
    TaxonomicOrder.CARNIVORA,
    SizeClass.MEDIUM, Trait.MOTILE, Trait.CARNIVORE, Trait.FLIGHT,
    Trait.BURROWING, Trait.ARMORED, Trait.VENOMOUS, Trait.REFLECTIVE_RETINA,
    Trait.CONSTRICTING_PUPILS, Trait.THICK_FUR, Trait.BLUBBER,
    Trait.COUNTERCURRENT_HEAT_EXCHANGE, Trait.CONCENTRATED_URINE,
    Trait.HUMP_FAT, Trait.TUSKS, Trait.ADAPTIVE_CAMOUFLAGE, Trait.HIBERNATION,
    Trait.SEASONAL_MIGRATION, Trait.ECHOLOCATION, Trait.ELECTRORECEPTION,
    Trait.HEAT_SENSING_PITS,
)
check(
    deriveMaintenanceRate(overbuiltChimeraBlueprint) >
        5.0 * speciesCatalog.first { it.id == "wolf" }.maintenanceRate
) { "Stacking many adaptations should create a severe energetic burden" }
check(EcosystemModel(speciesCatalog).altitudeMeters == 500.0) {
    "Notebook ecosystems should default to 500 m elevation"
}
check(speciesCatalog.first { it.id == "grass" }.blueprint.hasTag(EffectTag.WINTER_SURVIVAL)) {
    "Underground meristems should protect grass through freezing winters"
}
check(!speciesCatalog.first { it.id == "cactus" }.blueprint.hasTag(EffectTag.WINTER_SURVIVAL)) {
    "Cactus should not receive unearned winter hardiness"
}
check(speciesCatalog.first { it.id == "scorpion" }.climateNiche.minOptimalMoistureMm <
    speciesCatalog.first { it.id == "snake" }.climateNiche.minOptimalMoistureMm
) { "A waxy exoskeleton should make scorpions more drought tolerant" }
val hawkBirdPrey = speciesCatalog.first { it.id == "hawk" }.diet.keys.map { it.preyId }.toSet()
check(hawkBirdPrey.containsAll(setOf("sparrow", "pigeon", "quail"))) {
    "Flying hawks should be able to hunt the catalog's smaller birds"
}
val jerboaDefinition = speciesCatalog.first { it.id == "jerboa" }
check(Trait.BURROWING in jerboaDefinition.traits &&
    jerboaDefinition.climateNiche.maxOptimalMoistureMm == 140.0
) { "Jerboas should burrow, with tunnel flooding limiting very wet climates" }
val frogDefinition = speciesCatalog.first { it.id == "frog" }
check(frogDefinition.blueprint.hasTag(EffectTag.AERIAL_PREY_CAPTURE)) {
    "A projectile tongue should let frogs counter flying-insect evasion"
}
check(frogDefinition.diet.getValue(
    FoodSource("grasshopper", FoodResource.MOTILE_PREY),
).preference > speciesCatalog.first { it.id == "snake" }.diet.getValue(
    FoodSource("grasshopper", FoodResource.MOTILE_PREY),
).preference) { "Frogs should catch flying insects more effectively than ordinary ambush predators" }
check(setOf("bison", "caribou", "wildebeest", "bactrian_camel", "mammoth").all { id ->
    Trait.OPEN_PLAINS_LOCOMOTION in speciesCatalog.first { it.id == id }.traits
}) { "Open-country megafauna should be penalized by closed jungle canopy" }
check(insulationHeatStressMortalityRate(
    speciesCatalog.first { it.id == "mammoth" },
    ClimateDatumSample(28.0, 220.0, 250.0),
) > 0.5) { "Mammoth fur and subcutaneous fat should impose severe tropical heat stress" }
val sparseCoatedTropicalSpecies = setOf(
    "jaguar", "sloth", "gorilla", "capybara", "tapir",
    "lion", "zebra", "hyena", "kangaroo",
)
check(sparseCoatedTropicalSpecies.all { id ->
    Trait.SPARSE_COAT in speciesCatalog.first { it.id == id }.traits
}) { "Short-coated tropical mammals should use the sparse-coat trade-off" }
val rabbitFeeding = speciesCatalog.first { it.id == "rabbit" }.feeding!!
check(rabbitFeeding.mortalityRate < 0.10) {
    "Fed animals should have modest background mortality"
}
check(rabbitFeeding.starvationMortalityRate > 20.0 * rabbitFeeding.mortalityRate) {
    "Zero-food starvation should dominate background mortality"
}
val bearFeeding = speciesCatalog.first { it.id == "bear" }.feeding!!
check(abs(bearFeeding.assimilationEfficiency - 0.18) < 1e-12) {
    "Overlapping dietary niche traits must not stack global assimilation bonuses"
}
check(speciesCatalog.first { it.id == "deer" }.feeding!!.assimilationEfficiency > bearFeeding.assimilationEfficiency) {
    "Digestive physiology, not diet labels, should modify assimilation"
}
check(
    speciesCatalog.first { it.id == "dromedary" }.feeding!!.starvationMortalityRate <
        speciesCatalog.first { it.id == "oryx" }.feeding!!.starvationMortalityRate
) { "Hump fat should extend survival without food" }

check(hersfeldtIconicAnimalIds.size == 104) {
    "Every Hersfeldt climate classification must have iconic fauna"
}
check(hersfeldtIconicAnimalIds.values.flatten().all { animalId ->
    speciesCatalog.any { it.id == animalId }
}) { "Every iconic Hersfeldt animal must exist in the species catalog" }

check(speciesCatalog.any { it.autotrophy != null && it.feeding != null }) {
    "Catalog should exercise hybrid capability composition"
}
val fruitResourceEffect = Trait.FRUITING.effects
    .filterIsInstance<EdibleAs>()
    .single { it.resource == FoodResource.FRUIT }
check(fruitResourceEffect.preyLossFraction in 0.0..<1.0)
check(fruitResourceEffect.renewableProductionRate != null)
check(speciesCatalog.first { it.id == "berry_bush" }.let { fruiter ->
    val fruitDiet = speciesCatalog.first { it.id == "mouse" }
        .dietEntry(fruiter.id, FoodResource.FRUIT)!!
    fruitDiet.preyLossFraction == fruitResourceEffect.preyLossFraction &&
        fruitDiet.renewableProductionRate == fruitResourceEffect.renewableProductionRate
}) {
    "Fruiting should create a renewable, less-destructive food pathway"
}
check(speciesCatalog.first { it.id == "grasshopper" }.let { flyingPrey ->
    val foxPreference = speciesCatalog.first { it.id == "fox" }.maximumPreferenceFor(flyingPrey.id)
    val hawkPreference = speciesCatalog.first { it.id == "hawk" }.maximumPreferenceFor(flyingPrey.id)
    hawkPreference > foxPreference
}) {
    "Flight should favor flying predators"
}

check(
    speciesCatalog.first { it.id == "owl" }.climateNiche.minOptimalInsolationWm2 <
        speciesCatalog.first { it.id == "hawk" }.climateNiche.minOptimalInsolationWm2
) { "Reflective retinas should make owls functional at lower insolation than hawks" }
check(
    speciesCatalog.first { it.id == "wolf" }.climateNiche.minOptimalTemperatureC <
        speciesCatalog.first { it.id == "coyote" }.climateNiche.minOptimalTemperatureC
) { "Thick fur should lower the wolf's optimum temperature range" }
check(
    speciesCatalog.first { it.id == "shrub" }.climateNiche.minOptimalMoistureMm <
        speciesCatalog.first { it.id == "sundew" }.climateNiche.minOptimalMoistureMm
) { "Deep roots should require less precipitation than bog roots" }

println("Trait catalog contains " + speciesCatalog.size + " compiled species")
for (definition in speciesCatalog) {
    val capabilities = listOfNotNull(
        if (definition.autotrophy != null) "autotrophy" else null,
        if (definition.feeding != null) "feeding" else null,
    ).joinToString("+")
    val feedingCosts = definition.feeding?.let { feeding ->
        " demand=%.3f/y starvation=%.2f/y".format(
            feeding.metabolicDemandRate,
            feeding.starvationMortalityRate,
        )
    }.orEmpty()
    println(
        definition.id.padEnd(14) +
            definition.sizeClass.name.padEnd(11) +
            " individual biomass=" + "%8.4f".format(definition.individualBiomass) +
            " upkeep=" + "%.3f/y".format(definition.maintenanceRate) +
            feedingCosts +
            "  " + capabilities.padEnd(18) +
            " climate=" + "%.1f..%.1f C".format(
                definition.climateNiche.minOptimalTemperatureC,
                definition.climateNiche.maxOptimalTemperatureC,
            ) +
            " / >=%.0f mm".format(definition.climateNiche.minOptimalMoistureMm) +
            " / %.0f..%.0f W/m²".format(
                definition.climateNiche.minOptimalInsolationWm2,
                definition.climateNiche.maxOptimalInsolationWm2,
            ) +
            " diet=[" + definition.diet.keys.joinToString { source ->
                source.preyId + ":" + source.resource.name.lowercase()
            } + "]"
    )
}

val fullCatalogStructure = EcosystemModel(speciesCatalog)
check(
    fullCatalogStructure.consumerNicheOverlaps.getValue("rabbit").getValue("deer") >
        fullCatalogStructure.consumerNicheOverlaps.getValue("rabbit").getValue("wolf")
) {
    "Consumers sharing plant foods should exert stronger feeding interference"
}
val rabbitDeerOverlap = fullCatalogStructure.consumerNicheOverlaps
    .getValue("rabbit")
    .getValue("deer")
check(rabbitDeerOverlap in 0.75..0.95) {
    "Ground-foraging rabbits and high-browsing deer should overlap only partially: $rabbitDeerOverlap"
}
check(fullCatalogStructure.consumerNicheOverlaps
    .getValue("rabbit")
    .getValue("bear") == 0.0) {
    "Berry foliage and berry fruit must be separate competition resources"
}
check(Trait.RUMINANT_STOMACH in speciesCatalog.first { it.id == "deer" }.traits)
val rabbitDiet = speciesCatalog.first { it.id == "rabbit" }.diet
val deerDiet = speciesCatalog.first { it.id == "deer" }.diet
check(rabbitDiet.getValue(FoodSource("berry_bush", FoodResource.LOW_FOLIAGE)).preference >
    rabbitDiet.getValue(FoodSource("oak", FoodResource.CANOPY_FOLIAGE)).preference)
check(deerDiet.getValue(FoodSource("oak", FoodResource.CANOPY_FOLIAGE)).preference >
    deerDiet.getValue(FoodSource("berry_bush", FoodResource.LOW_FOLIAGE)).preference)
val expectedMinusculeSessileGuild = speciesCatalog
    .filter { it.sizeClass == SizeClass.MINUSCULE && it.isSessileProducer() }
    .map { it.id }
    .toSet()
check(
    fullCatalogStructure.sessileProducerGroups
        .getValue(SizeClass.MINUSCULE)
        .map { it.id }
        .toSet() == expectedMinusculeSessileGuild
) {
    "Same-sized sessile producers should share one carrying-capacity guild"
}

val exampleSpeciesIds = setOf("grass", "rabbit", "wolf")
val species = speciesCatalog.filter { it.id in exampleSpeciesIds }
validateFoodWeb(species)
val ecosystem = EcosystemModel(species, seasonsEnabled = true, landArea = 1_000_000.0)

val initialBiomass: Biomass = mapOf(
    "grass" to 250_000.0,
    "rabbit" to 40_000.0,
    "wolf" to 8_000.0,
)


In [ ]:
val suitabilitySpotCheckElevationMeters = 500.0
val suitabilityExampleClimates = linkedMapOf(
    "oceanic_temperate" to oceanicTemperateClimate,
) + sampleClimates

println("Species suited for every month at ${suitabilitySpotCheckElevationMeters.toInt()} m:")
for ((climateName, climate) in suitabilityExampleClimates) {
    val suited = speciesCatalog.filter { species ->
        climate.months.all { sample ->
            isSpeciesSuitedTo(species, sample, suitabilitySpotCheckElevationMeters)
        }
    }
    println("%-18s %3d: %s".format(
        climateName,
        suited.size,
        suited.joinToString { it.id },
    ))
}


## Positivity-safe stochastic simulation and extinction

The shared simulator uses RK4 substeps, multiplicative process noise, and a two-individual extinction threshold. The probes below guard starvation, climate envelopes, renewable foods, and competition behavior.


In [ ]:
val starvationProbeSpecies = listOf(speciesCatalog.first { it.id == "rabbit" })
val starvationProbe = simulate(
    model = EcosystemModel(
        species = starvationProbeSpecies,
        seasonsEnabled = false,
        landArea = 1_000.0,
    ),
    initial = mapOf("rabbit" to 10.0),
    years = 3.0,
    dt = 1.0 / 365.0,
    sampleEverySteps = 365,
    noise = PopulationNoise.NONE,
)
check(starvationProbe.last().biomass.getValue("rabbit") == 0.0) {
    "A consumer with no available food should starve to extinction within three years"
}

/** Deterministic single-producer probe used to guard the sample biome envelopes. */
fun finalProducerBiomassInClimate(speciesId: String, climate: ClimateDatum): Double {
    val definition = speciesCatalog.first { it.id == speciesId }
    val probeModel = EcosystemModel(
        species = listOf(definition),
        seasonsEnabled = true,
        landArea = 10_000.0,
        climate = climate,
    )
    return simulate(
        model = probeModel,
        initial = mapOf(speciesId to 4.0 * definition.individualBiomass),
        years = 120.0,
        dt = 1.0 / 120.0,
        sampleEverySteps = 120 * 120,
        noise = PopulationNoise.NONE,
    ).last().biomass.getValue(speciesId)
}

check(finalProducerBiomassInClimate("grass", tundraClimate) > 0.0) {
    "Underground grass meristems should persist through the tundra climate"
}
check(finalProducerBiomassInClimate("dwarf_willow", tundraClimate) > 0.0) {
    "A prostrate woody producer should persist in tundra"
}
check(finalProducerBiomassInClimate("spruce", borealClimate) > 0.0) {
    "Needle-leaved evergreen trees should persist in boreal climate"
}
check(setOf("grass", "clover", "wildflowers").all { speciesId ->
    finalProducerBiomassInClimate(speciesId, desertClimate) == 0.0
}) { "Temperate meadow plants should not establish in the desert sample" }
check(finalProducerBiomassInClimate("cactus", desertClimate) > 0.0) {
    "The desert sample should support a sparse cactus population"
}
check(finalProducerBiomassInClimate("dwarf_willow", desertClimate) == 0.0) {
    "A prostrate tundra shrub should not persist in a hot desert"
}

val renewableProbeSpecies = speciesCatalog.filter {
    it.id in setOf("berry_bush", "mouse", "rabbit", "deer", "bear")
}
val renewableProbeState = mapOf(
    "berry_bush" to 1_000.0,
    "mouse" to 100.0,
    "rabbit" to 100.0,
    "deer" to 100.0,
    "bear" to 100.0,
)
val renewableProbeModel = EcosystemModel(
    renewableProbeSpecies,
    seasonsEnabled = false,
    landArea = 100.0,
)
val renewableProbeFluxes = feedingFluxes(
    renewableProbeState,
    renewableProbeModel,
)
for ((resource, productionRate) in mapOf(
    FoodResource.FRUIT to 0.30,
    FoodResource.SEED to 0.08,
)) {
    val source = FoodSource("berry_bush", resource)
    val totalIngestion = renewableProbeFluxes
        .filter { it.source == source }
        .sumOf { it.ingested }
    check(totalIngestion <= renewableProbeState.getValue(source.preyId) * productionRate + 1e-9) {
        "Consumers exceeded renewable production for $source"
    }
}
check(renewableProbeFluxes.any { flux ->
    flux.feeder == "bear" &&
        flux.source.resource == FoodResource.MOTILE_PREY &&
        flux.ingested > 0.0
}) { "A bear should redirect unmet renewable-food demand toward animal prey" }
val probeBear = renewableProbeSpecies.first { it.id == "bear" }
val probeBearFeeding = probeBear.feeding!!
val probeBearInterferenceLoad = probeBearFeeding.interference *
    renewableProbeModel.consumerNicheOverlaps.getValue("bear").entries.sumOf {
        (competitorId, overlap) ->
        overlap * renewableProbeState.getValue(competitorId)
    } / renewableProbeModel.landArea
for (flux in renewableProbeFluxes.filter {
    it.feeder == "bear" && it.source.resource == FoodResource.MOTILE_PREY
}) {
    val diet = probeBear.diet.getValue(flux.source)
    val preyDensity = renewableProbeState.getValue(flux.source.preyId) /
        renewableProbeModel.landArea
    val signal = diet.preference *
        (preyDensity / diet.halfSaturation).pow(probeBearFeeding.responseExponent)
    val pathwayCapacity = probeBearFeeding.maxConsumptionRate *
        renewableProbeState.getValue("bear") * signal /
        (1.0 + signal + probeBearInterferenceLoad)
    check(flux.ingested <= pathwayCapacity * (1.0 + 1e-10) + 1e-9) {
        "Diet switching bypassed the Type III pathway ceiling for ${flux.source}"
    }
}

val abundantHerbivoreSpecies = speciesCatalog.filter {
    it.id in setOf("grass", "oak", "berry_bush", "rabbit", "deer")
}
val abundantHerbivoreModel = EcosystemModel(
    species = abundantHerbivoreSpecies,
    seasonsEnabled = false,
    landArea = 1_000.0,
    sessileCarryingCapacityDensities = mapOf(
        SizeClass.MINUSCULE to 1_000.0,
        SizeClass.MEDIUM to 1_000.0,
        SizeClass.GIANT to 1_000.0,
    ),
)
val abundantHerbivoreHistory = simulate(
    model = abundantHerbivoreModel,
    initial = mapOf(
        "grass" to 500_000.0,
        "oak" to 500_000.0,
        "berry_bush" to 500_000.0,
        "rabbit" to 100.0,
        "deer" to 1_000.0,
    ),
    years = 120.0,
    dt = 1.0 / 120.0,
    sampleEverySteps = 120,
    noise = PopulationNoise.NONE,
)
val abundantHerbivoreFinal = abundantHerbivoreHistory.last().biomass
check(abundantHerbivoreFinal.getValue("rabbit") / abundantHerbivoreSpecies.first { it.id == "rabbit" }.individualBiomass >= minimumViableIndividuals)
check(abundantHerbivoreFinal.getValue("deer") / abundantHerbivoreSpecies.first { it.id == "deer" }.individualBiomass >= minimumViableIndividuals) {
    "Diet overlap alone must not eliminate deer while food is superabundant"
}


## Stable seasonal cycle

Start from arbitrary biomass and run long enough for the assembled ecosystem to find its own attractor. The plot normalizes each species by its observed mean over the final twenty years so differently sized species remain visible on the same scale. This reference is a measurement, not an input to the model.

In [ ]:
val seasonalHistory = simulate(
    model = ecosystem,
    initial = initialBiomass,
    years = 200.0,
)
val stableWindowStart = 180.0
val stableWindow = seasonalHistory.filter { it.year >= stableWindowStart }
val referenceBiomass: Biomass = species.associate { definition ->
    definition.id to stableWindow.map { it.biomass.getValue(definition.id) }.average()
}
val persistentReference = referenceBiomass.filterValues { it > 0.0 }
val lostSpecies = referenceBiomass.filterValues { it == 0.0 }.keys
if (lostSpecies.isNotEmpty()) println("Lost during assembly: ${lostSpecies.joinToString()}")

data class PlotPoint(
    val year: Double,
    val species: String,
    val logBiomass: Double,
)

val plotBiomassFloor = 1e-12

fun logPlotData(
    history: List<SimulationPoint>,
    speciesIds: Collection<String>,
    fromYear: Double,
): List<PlotPoint> = history
    .filter { it.year >= fromYear }
    .flatMap { point ->
        speciesIds.map { species ->
            PlotPoint(
                year = point.year,
                species = species,
                logBiomass = log10(
                    point.biomass.getValue(species).coerceAtLeast(plotBiomassFloor)
                ),
            )
        }
    }

val stableCyclePlotData = logPlotData(
    seasonalHistory,
    persistentReference.keys,
    stableWindowStart,
)

plot {
    layout.title = "Stable annual ecosystem cycle"
    layout.size = 1000 to 500
    line {
        x(stableCyclePlotData.map { it.year }, name = "year")
        y(stableCyclePlotData.map { it.logBiomass }, name = "log10 biomass")
        color(stableCyclePlotData.map { it.species }, name = "species")
    }
}

In [ ]:
data class StabilityMetrics(
    val species: String,
    val mean: Double,
    val minimum: Double,
    val maximum: Double,
    val coefficientOfVariation: Double,
    val minimumToMean: Double,
)

fun stabilityMetrics(
    history: List<SimulationPoint>,
    speciesIds: Collection<String>,
    fromYear: Double,
): List<StabilityMetrics> {
    val window = history.filter { it.year >= fromYear }
    require(window.isNotEmpty())

    return speciesIds.map { species ->
        val values = window.map { it.biomass.getValue(species) }
        val mean = values.average()
        val standardDeviation = sqrt(values.sumOf { (it - mean).pow(2) } / values.size)
        StabilityMetrics(
            species = species,
            mean = mean,
            minimum = values.minOrNull()!!,
            maximum = values.maxOrNull()!!,
            coefficientOfVariation = if (mean > 0.0) standardDeviation / mean else 0.0,
            minimumToMean = if (mean > 0.0) values.minOrNull()!! / mean else 0.0,
        )
    }
}

val stableMetrics = stabilityMetrics(
    history = seasonalHistory,
    speciesIds = referenceBiomass.keys,
    fromYear = stableWindowStart,
)

println("species    mean      min       max       CV       min/mean")
stableMetrics.forEach { metric ->
    println(
        "%-8s  %.4f    %.4f    %.4f    %.3f    %.3f".format(
            metric.species, metric.mean, metric.minimum, metric.maximum,
            metric.coefficientOfVariation, metric.minimumToMean,
        )
    )
}

## Recovery from a major disruption

After allowing the annual cycle to settle, remove 65% of the rabbits at year 10. No population floor or immigration rescues them: recovery comes entirely from the local growth and feeding equations.

In [ ]:
val disturbanceYear = 10.0
val settledState = seasonalHistory.last().biomass
val disruptedHistory = simulate(
    model = ecosystem,
    initial = settledState,
    years = 50.0,
    disturbances = listOf(
        Disturbance(
            year = disturbanceYear,
            remainingFraction = mapOf("rabbit" to 0.35),
        )
    ),
)

val disruptionPlotData = logPlotData(disruptedHistory, persistentReference.keys, fromYear = 5.0)

plot {
    layout.title = "Recovery after removing 65% of rabbits at year 10"
    layout.size = 1000 to 500
    line {
        x(disruptionPlotData.map { it.year }, name = "year")
        y(disruptionPlotData.map { it.logBiomass }, name = "log10 biomass")
        color(disruptionPlotData.map { it.species }, name = "species")
    }
}

In [ ]:
fun sustainedRecoveryTime(
    history: List<SimulationPoint>,
    species: String,
    target: Double,
    afterYear: Double,
    tolerance: Double = 0.10,
    holdYears: Double = 2.0,
): Double? {
    val after = history.filter { it.year >= afterYear }
    for ((index, candidate) in after.withIndex()) {
        val endYear = candidate.year + holdYears
        val holdWindow = after.subList(index, after.size).takeWhile { it.year <= endYear }
        val spansHoldPeriod = holdWindow.lastOrNull()?.year?.let { it >= endYear - 0.02 } == true
        val staysInBand = holdWindow.all {
            abs(it.biomass.getValue(species) / target - 1.0) <= tolerance
        }
        if (spansHoldPeriod && staysInBand) return candidate.year - afterYear
    }
    return null
}

println("species    minimum after shock    sustained recovery to +/-10%")
for ((species, target) in persistentReference) {
    val afterShock = disruptedHistory.filter { it.year >= disturbanceYear }
    val minimumFraction = afterShock.minOf { it.biomass.getValue(species) / target }
    val recovery = sustainedRecoveryTime(
        disruptedHistory, species, target, afterYear = disturbanceYear
    )
    val recoveryText = recovery?.let { "%.1f years".format(it) } ?: "not within simulation"
    println("%-8s   %.3f                  %s".format(species, minimumFraction, recoveryText))
}

## Random ecosystem assemblies

Each trial samples already compiled species without changing their traits or derived parameters. One autotrophic species is guaranteed as an energy source; all remaining slots are sampled blindly. Diet entries may point to absent prey, feeding species may starve, and sampled species may exclude one another.

The test uses a three-day integration step, checks every biomass for finite nonnegative values, and calls a species persistent when its final twenty-year mean exceeds the reporting threshold.

In [ ]:
fun randomSpeciesSample(
    catalog: List<SpeciesDefinition>,
    count: Int,
    random: Random,
): List<SpeciesDefinition> {
    require(count in 1..catalog.size)
    val autotroph = catalog.filter { it.autotrophy != null }.random(random)
    return listOf(autotroph) + (catalog - autotroph).shuffled(random).take(count - 1)
}

fun randomInitialBiomass(
    model: EcosystemModel,
    random: Random,
): Biomass = model.species.associate { definition ->
    val initialDensity = when {
        definition.isSessileProducer() -> {
            val guildSize = model.sessileProducerGroups
                .getValue(definition.sizeClass)
                .size
            model.sessileCarryingCapacityDensity(definition.sizeClass, year = 0.0) *
                random.nextDouble(0.10, 0.65) / guildSize
        }
        definition.autotrophy != null ->
            definition.autotrophy.baseCarryingCapacity *
                random.nextDouble(0.10, 0.65)
        else -> random.nextDouble(0.002, 0.060)
    }
    val densityBasedBiomass = initialDensity * model.landArea
    val foundingBiomass = 4.0 * definition.individualBiomass
    definition.id to maxOf(densityBasedBiomass, foundingBiomass)
}

data class RandomAssemblyResult(
    val seed: Int,
    val selected: List<String>,
    val persistent: List<String>,
    val lost: List<String>,
    val maximumPersistentCv: Double,
)

fun runRandomAssembly(
    catalog: List<SpeciesDefinition>,
    speciesCount: Int,
    seed: Int,
    years: Double = 120.0,
    measurementYears: Double = 20.0,
): RandomAssemblyResult {
    val random = Random(seed)
    val selected = randomSpeciesSample(catalog, speciesCount, random)
    validateFoodWeb(selected)
    val model = EcosystemModel(selected, seasonsEnabled = true, landArea = 100_000.0)
    val history = simulate(
        model,
        randomInitialBiomass(model, random),
        years = years,
        dt = 1.0 / 120.0,
        sampleEverySteps = 4,
        random = random,
    )
    check(history.all { point ->
        point.biomass.values.all { it.isFinite() && it >= 0.0 }
    }) { "Assembly " + seed + " produced invalid biomass" }

    val metrics = stabilityMetrics(
        history,
        selected.map { it.id },
        fromYear = years - measurementYears,
    )
    val persistent = metrics.filter { it.mean > 0.0 }
    val persistentIds = persistent.map { it.species }.sorted()
    val selectedIds = selected.map { it.id }.sorted()
    return RandomAssemblyResult(
        seed,
        selectedIds,
        persistentIds,
        selectedIds - persistentIds.toSet(),
        persistent.maxOfOrNull { it.coefficientOfVariation } ?: 0.0,
    )
}

val randomAssemblyResults = (0..<12).map { index ->
    runRandomAssembly(speciesCatalog, speciesCount = 10, seed = 10_000 + index)
}

println("seed    selected  persistent  lost  max persistent CV")
for (result in randomAssemblyResults) {
    println("%-7d %-9d %-11d %-5d %.3f".format(
        result.seed,
        result.selected.size,
        result.persistent.size,
        result.lost.size,
        result.maximumPersistentCv,
    ))
    println("  selected:   " + result.selected.joinToString())
    println("  persistent: " + result.persistent.joinToString())
    if (result.lost.isNotEmpty()) println("  lost:       " + result.lost.joinToString())
}

check(randomAssemblyResults.all { result ->
    result.persistent.isNotEmpty() && result.selected.any { id ->
        speciesCatalog.first { it.id == id }.autotrophy != null
    }
})

In [ ]:
// Change this seed to draw another reproducible five-species ecosystem.
val randomFiveSeed = Random.nextInt()
val randomFiveRandom = Random(randomFiveSeed)
val randomFiveSpecies = randomSpeciesSample(
    catalog = speciesCatalog,
    count = 5,
    random = randomFiveRandom,
)
val randomFiveModel = EcosystemModel(randomFiveSpecies, seasonsEnabled = true, landArea = 100_000.0)
val randomFiveInitial = randomInitialBiomass(randomFiveModel, randomFiveRandom)
val randomFiveHistory = simulate(
    model = randomFiveModel,
    initial = randomFiveInitial,
    years = 120.0,
    dt = 1.0 / 120.0,
    sampleEverySteps = 4,
    random = randomFiveRandom,
)

println("Random five-species sample: " + randomFiveSpecies.joinToString { it.id })

val randomFivePlotData = logPlotData(
    randomFiveHistory,
    randomFiveSpecies.map { it.id },
    fromYear = 0.0,
)

plot {
    layout.title = "Random five-species ecosystem (seed " + randomFiveSeed + ")"
    layout.size = 1000 to 500
    line {
        x(randomFivePlotData.map { it.year }, name = "year")
        y(randomFivePlotData.map { it.logBiomass }, name = "log10 biomass")
        color(randomFivePlotData.map { it.species }, name = "species")
    }
}

In [ ]:
//val selectedSpecies = speciesCatalog.filter { it.id in listOf("grass", "oak", "berry_bush", "deer", "rabbit", "vole", "fox", "wolf", "ibex", "horse", "zebra", "bear", "lion", "tiger", "giant_panda", "bamboo" ) }
val selectedSpecies = speciesCatalog.filter { (Trait.AQUATIC !in it.traits && Random.nextDouble() < 0.5) || it.id in listOf("grass") }
//val selectedSpecies = speciesCatalog
val selectedSessileCarryingCapacityDensities = mapOf(
    SizeClass.MINUSCULE to 500.00,
    SizeClass.GIANT to 100.80,
    SizeClass.MEDIUM to 100.0,
)
val selectedClimate =
    listOf(oceanicTemperateClimate to "oceanic temperate", desertClimate to "desert", savannaClimate to "savanna", jungleClimate to "jungle", borealClimate to "boreal", tundraClimate to "tundra", iceCapClimate to "ice cap")
    .filter { it.second != "ice cap" }
    .random()
val selectedLandArea = 100000.0
val selectedModel = EcosystemModel(
    species = selectedSpecies,
    seasonsEnabled = true,
    landArea = selectedLandArea,
    sessileCarryingCapacityDensities = selectedSessileCarryingCapacityDensities,
    climate = selectedClimate.first
)
val selectedInitial = randomInitialBiomass(selectedModel, Random)
val selectedHistory = simulate(
    model = selectedModel,
    initial = selectedInitial,
    years = 200.0,
    dt = 1.0 / 120.0,
    sampleEverySteps = 4,
    noise = PopulationNoise(
        environmentalVolatility = 0.2,
        speciesVolatility = 0.3,
    ),
)

println("Selected climate: ${selectedClimate.second}")
println("Selected-species sample: " + selectedSpecies.joinToString { it.id })

println("Survived: ${selectedHistory.last().biomass.filter { it.value > 0 }.map { it.key }}")
println("Extinct: ${selectedHistory.last().biomass.filter { it.value == 0.0 }.map { it.key }}")

val selectedPlotData = logPlotData(
    selectedHistory,
    selectedSpecies.map { it.id },
    fromYear = 0.0,
)

plot {
    layout.title = "Custom ecosystem"
    layout.size = 1000 to 500
    line {
        x(selectedPlotData.map { it.year }, name = "year")
        y(selectedPlotData.map { it.logBiomass }, name = "log10 biomass")
        color(selectedPlotData.map { it.species }, name = "species")
    }
}

## Useful experiments

- Change the sample count, sample size, or seeds in randomAssemblyResults.
- Change seasonalAmplitude to control the environmental cycle strength.
- Move responseExponent toward 1.0 to compare Type III feeding with Type II feeding.
- Reduce densityDependence to weaken disease, territoriality, and other self-crowding losses.
- Reduce interference to weaken intake obstruction from consumers with overlapping diets.
- Inspect consumerNicheOverlaps to see which feeding guilds interfere most strongly when food is scarce.
- Change FRUITING or SEED_BEARING renewableProductionRate values to alter annual renewable-food supply.
- Compare GROUND_FORAGING, HIGH_BROWSING, and RUMINANT_STOMACH against plants exposing LOW_FOLIAGE, CANOPY_FOLIAGE, and COARSE_GRASS.
- Change sessileCarryingCapacityDensities to alter the shared producer capacity of each size class.
- Add trait-only species blueprints and inspect their compiled parameters and diets.
- Stack concrete adaptations such as THICK_FUR, BLUBBER, DEEP_ROOTS, WAXY_CUTICLE, or CONCENTRATED_URINE and inspect the resulting climate niche.
- Replace oceanicTemperateClimate with any tile ClimateDatum from ClimateSimulation.
- Use sampleClimates["desert"], ["savanna"], ["jungle"], ["boreal"], ["tundra"], or ["ice_cap"] for reference-biome experiments.
- Compare combinations of REFLECTIVE_RETINA, LARGE_EYES, CONSTRICTING_PUPILS, and UV_FILTERING_LENSES across insolation regimes.
- Sample the entire catalog to study competition, omnivory, and trophic cascades.
- Use hersfeldtIconicAnimalIds[climateId] as a climate-specific candidate pool; these are alternatives associated with the zone, not a claim that every listed animal coexists.

Extinction occurs when a species has fewer than `minimumViableIndividuals` (two by default), computed from total biomass and the species' size-derived `individualBiomass`. Pass a different `minimumIndividuals` to `simulate` to test another demographic cutoff.